In [ ]:
import numpy as np

In [ ]:
def norma(x,p):
    max = 0
    res = 0
    if x.ndim ==1:
      if p == 'inf':
        for i in range(len(x)):
          if max < abs(x[i]):
            max = abs(x[i])
        return max

      for i in range(len(x)):
          res+= abs(x[i])**p
      return res**(1/p)

    n = x.shape[1]
    resultado = np.zeros(n)

    for j in range(n):
      resultado[j] = norma(x[:,j],p)
    return resultado

"""
def norma(x, p):
    if x.ndim == 1:
        if p == 'inf':
            return max(abs(v) for v in x)
        res = 0
        for v in x:
            res += abs(v)**p
        return res**(1/p)

    n = x.shape[1]
    resultado = np.zeros(n)

    if p == 'inf':
        for j in range(n):
            resultado[j] = max(abs(v) for v in x[:, j])
    else:
        for j in range(n):
            res = 0
            for v in x[:, j]:
                res += abs(v)**p
            resultado[j] = res**(1/p)

    return resultado
"""


def normaliza(X,p):
    res = []
    for v in X:
        v_norm = v/norma(v,p)
        res.append(v_norm)
    return res

#"""
def normaMatMC(A,q,p,Np):
  n = A.shape[1]
  X = np.random.rand(n,Np)
  X_norm = X / norma(X,p)
  Y = A@X_norm
  norms = norma(Y,q)
  max_norm = 0
  idx = 0
  for i in range(Np):
    if max_norm < norms[i]:
      max_norm = norms[i]
      idx = i
  return max_norm, X_norm[:,idx]
#"""

"""
def normaMatMC(A,q,p,Np):
  n = A.shape[1]
  X = np.random.randn(n,Np)
  X_norm = X / norma(X,p)
  Y = A@X_norm
  norms = norma(Y,q)
  idx = np.argmax(norms)
  max_norm = norms[idx]
  return max_norm, X_norm[:,idx]
"""

"""
def normaExacta(A,p=[1,'inf']):
  norm_1 = 0
  norm_inf = 0
  fil, col = A.shape
  for j in range(col):
   suma = 0
   for i in range(fil):
     suma += abs(A[i][j])
   if norm_1 < suma:
     norm_1 = suma

  for i in range(fil):
    suma = 0
    for j in range(col):
      suma += abs(A[i][j])
    if norm_inf < suma:
      norm_inf = suma

  if p == 1:
    return norm_1

  if p == 'inf':
    return norm_inf

  return [norm_1, norm_inf]
"""

def norma_1_inf(hasta_n, hasta_m, A, flag):
  comparador = 0
  for i in hasta_n:
    suma = 0
    for j in hasta_m:
      if flag:
        suma += abs(A[i][j])
      else:
        suma += abs(A[j][i])
    if comparador < suma:
      comparador = suma
  return comparador


def normaExacta(A,p=[1,'inf']):
  fil, col = A.shape

  if p==1:
    return norma_1_inf(range(col), range(fil), A, False)

  if p=='inf':
    return norma_1_inf(range(fil), range(col), A, True)

  if p==[1,'inf']:
    return [norma_1_inf(range(col), range(fil), A, False), norma_1_inf(range(fil), range(col), A, True)]




def condMC(A, p,Np):
  norma_1_y_inf_A = normaExacta(A,'inf')
  A_inv = np.linalg.inv(A)
  norma_1_y_inf_A_inv = normaExacta(A_inv,'inf')
  normaA = 0
  normaA_inv = 0
  if p == 1:
    normaA = norma_1_y_inf_A[0]
    normaA_inv = norma_1_y_inf_A_inv[0]

  elif p == 'inf':
    normaA = norma_1_y_inf_A[1]
    normaA_inv = norma_1_y_inf_A_inv[1]
  else:
    normaA = normaMatMC(A,p,p,Np)[0]
    normaA_inv = normaMatMC(A_inv,p,p,Np)[0]

  return normaA * normaA_inv

"""
def condMC(A, p):
  A_inv = np.linalg.inv(A)

  normaA = normaMatMC(A, p, p, 1000)[0]
  normaA_inv = normaMatMC(A_inv, p, p, 1000)[0]

  return normaA * normaA_inv
"""

def condExacta(A,p):
  norm_A = normaExacta(A,p)
  A_inv = np.linalg.inv(A)
  norm_A_inv = normaExacta(A_inv,p)
  return norm_A * norm_A_inv


In [ ]:
# Tests L03-Normas

# Tests norma
assert(np.allclose(norma(np.array([1,1]),2),np.sqrt(2)))
assert(np.allclose(norma(np.array([1]*10),2),np.sqrt(10)))
assert(norma(np.random.rand(10),2)<=np.sqrt(10))
assert(norma(np.random.rand(10),2)>=0)

# Tests normaliza
# Tests normaliza
for x in normaliza([np.array([1]*k) for k in range(1,11)],2):
    assert(np.allclose(norma(x,2),1))
for x in normaliza([np.array([1]*k) for k in range(2,11)],1):
    print(not np.allclose(norma(x,2),1) )
for x in normaliza([np.random.rand(k) for k in range(1,11)],'inf'):
    assert( np.allclose(norma(x,'inf'),1) )
# Test normaMC

nMC = normaMatMC(A=np.eye(2),q=2,p=1,Np=100000)
assert(np.allclose(nMC[0],1,atol=1e-3))
assert(np.allclose(np.abs(nMC[1][0]),1,atol=1e-3) or np.allclose(np.abs(nMC[1][1]),1,atol=1e-3))
assert(np.allclose(np.abs(nMC[1][0]),0,atol=1e-3) or np.allclose(np.abs(nMC[1][1]),0,atol=1e-3))

nMC = normaMatMC(A=np.eye(2),q=2,p='inf',Np=100000)
assert(np.allclose(nMC[0],np.sqrt(2),atol=1e-3))
assert(np.allclose(np.abs(nMC[1][0]),1,atol=1e-3) and np.allclose(np.abs(nMC[1][1]),1,atol=1e-3))


A = np.array([[1,2],[3,4]])
nMC = normaMatMC(A=A,q='inf',p='inf',Np=1000000)
assert(np.allclose(nMC[0],normaExacta(A,'inf'),rtol=2e-1))

A = np.array([[1,1],[0,1]])
A_ = np.linalg.solve(A,np.eye(A.shape[0]))
normaA = normaMatMC(A,2,2,10000)
normaA_ = normaMatMC(A_,2,2,10000)
condA = condMC(A,2,10000)
assert(np.allclose(normaA[0]*normaA_[0],condA,atol=1e-3))

A = np.array([[3,2],[4,1]])
A_ = np.linalg.solve(A,np.eye(A.shape[0]))
normaA = normaMatMC(A,2,2,10000)
normaA_ = normaMatMC(A_,2,2,10000)
condA = condMC(A,2,10000)
assert(np.allclose(normaA[0]*normaA_[0],condA,atol=1e-3))

# Test condExacta

A = np.random.rand(10,10)
A_ = np.linalg.solve(A,np.eye(A.shape[0]))
normaA = normaExacta(A,1)
normaA_ = normaExacta(A_,1)
condA = condExacta(A,1)
assert(np.allclose(normaA*normaA_,condA))

A = np.random.rand(10,10)
A_ = np.linalg.solve(A,np.eye(A.shape[0]))
normaA = normaExacta(A,'inf')
normaA_ = normaExacta(A_,'inf')
condA = condExacta(A,'inf')
assert(np.allclose(normaA*normaA_,condA))




True
True
True
True
True
True
True
True
True
